In [1]:
import json
import os
import chromadb
from chromadb.utils import embedding_functions

# 1. Read the JSON file
filepath = "data/shl_catalog.json" # Adjust this if your json is in the root folder instead
with open(filepath, 'r', encoding='utf-8') as file:
    raw_catalog = json.load(file)
    
documents = []
metadatas = []
ids = []

# 2. Clean and format the data
for item in raw_catalog:
    # Skip items that are missing essential info
    if not item.get("name") or not item.get("description"):
        continue
        
    job_levels = ", ".join(item.get("job_levels", []))
    languages = ", ".join(item.get("languages", []))
    keys = ", ".join(item.get("keys", []))
    
    # Create the text we want the AI to "read"
    rich_text = (
        f"Assessment Name: {item['name']}\n"
        f"Description: {item['description']}\n"
        f"Target Job Levels: {job_levels}\n"
        f"Measured Skills/Keys: {keys}\n"
        f"Supported Languages: {languages}\n"
        f"Is Adaptive: {item.get('adaptive', 'no')}\n"
        f"Duration: {item.get('duration', 'Not specified')}"
    )
    documents.append(rich_text)
    
    # Save the clean data to return to the user later
    metadatas.append({
        "name": item["name"],
        "url": item.get("link", ""),
        "duration": item.get("duration", "Not specified"),
        "test_type": "K" if "Knowledge & Skills" in keys else "P" if "Personality" in keys else "A" 
    })
    
    ids.append(str(item["entity_id"]))

print(f"Cleaned {len(documents)} assessments. Building database now...")

# 3. Create the local Vector Database inside your data folder
chroma_client = chromadb.PersistentClient(path="./data/shl_chroma_db")
embedding_model = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

collection = chroma_client.get_or_create_collection(
    name="shl_assessments",
    embedding_function=embedding_model
)

# 4. Save everything into the database
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("SUCCESS! Database built and saved locally.")

Cleaned 377 assessments. Building database now...


c:\Users\kul_m\shl-agent-assignment\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\kul_m\shl-agent-assignment\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kul_m\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator

SUCCESS! Database built and saved locally.


In [2]:
# 1. Reconnect to the database collection we just built
import chromadb
from chromadb.utils import embedding_functions

chroma_client = chromadb.PersistentClient(path="./data/shl_chroma_db")
embedding_model = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = chroma_client.get_collection(name="shl_assessments", embedding_function=embedding_model)

# 2. Define a realistic search query from a hiring manager
test_query = "I need an advanced assessment for a Java developer who knows microservices and cloud."

# 3. Query the top 3 matches
results = collection.query(
    query_texts=[test_query],
    n_results=3
)

# 4. Print out the names and metadata of what it found
print("--- SEMANTIC SEARCH TEST RESULTS ---")
for i in range(len(results['ids'][0])):
    print(f"\nMatch #{i+1} (ID: {results['ids'][0][i]}):")
    print(f"Name: {results['metadatas'][0][i]['name']}")
    print(f"URL: {results['metadatas'][0][i]['url']}")
    print(f"Duration: {results['metadatas'][0][i]['duration']}")

--- SEMANTIC SEARCH TEST RESULTS ---

Match #1 (ID: 4033):
Name: Java Web Services (New)
URL: https://www.shl.com/products/product-catalog/view/java-web-services-new/
Duration: 8 minutes

Match #2 (ID: 4091):
Name: Microservices (New)
URL: https://www.shl.com/products/product-catalog/view/microservices-new/
Duration: 7 minutes

Match #3 (ID: 4034):
Name: Core Java (Advanced Level) (New)
URL: https://www.shl.com/products/product-catalog/view/core-java-advanced-level-new/
Duration: 13 minutes
